# Two-dimensional continuous systems — exercise lab

This laboratory accompanies [the chapter](https://mvreeuwijk.github.io/chaosbook/cont2d.html). It runs
entirely in your browser (Python via Pyodide) — nothing to install, and any
changes you make are yours alone: re-opening the lab from the chapter resets it.

The full text of the exercises is in the
[chapter's exercise section](https://mvreeuwijk.github.io/chaosbook/cont2d.html#exercises); this notebook
gives you a running start on each one.

In [ ]:
# Run this cell first: it installs the book's package into the
# in-browser Python (takes a few seconds, needs to run once per visit).
%pip install -q chaosbook ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider
import chaosbook as cb

## Phase portraits at full parameter freedom

The chapter's interactive figures step through a precomputed grid of parameter
values. Here every slider move recomputes the portrait: direction field,
nullclines (blue: $\dot x=0$, red dashed: $\dot y=0$), fixed points with their
type, and two trajectories. Try the van der Pol oscillator through $\mu = 2$
(where the origin changes from spiral to node), or edit `rhs` to explore any
planar system of your own.

In [ ]:
def rhs(t, s, mu=1.0):                 # van der Pol; replace by your own system
    x, y = s
    return [y, -x + mu * (1 - x**2) * y]

def explorer(mu=1.0, lim=3.0):
    fig, ax = plt.subplots(figsize=(5, 5))
    cb.phase_portrait(rhs, (-lim, lim), (-lim, lim), ax=ax, n=21, style="stream",
                      nullclines=True, trajectories=[[0.1, 0], [lim * 0.9, 0]],
                      tmax=40, mu=mu)
    for p in cb.fixed_points_nd(rhs, [[0.0, 0.0], [1.0, 1.0], [-1.0, -1.0]], mu=mu):
        ev, kind = cb.classify_fixed_point(cb.jacobian(rhs, p, mu=mu))
        ax.plot(*p, "ko", mfc="w", ms=8)
        ax.set_title(f"{kind}: sigma = {np.round(ev, 3)}")
    plt.show()

interact(explorer, mu=FloatSlider(1.0, min=0.0, max=4.0, step=0.1),
         lim=FloatSlider(3.0, min=1.0, max=6.0, step=0.5));

## Exercise: fixed points and linearization

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/cont2d.html#fixed-points-and-linearization).
Below the first system is done with sympy (symbolic fixed points and Jacobian),
then classified and plotted. Repeat for the other two.

In [ ]:
x, y = sp.symbols("x y", real=True)
F = sp.Matrix([x - y, x**2 - 4])               # system (a); edit for (b) and (c)
fps = sp.solve(list(F), [x, y], dict=True)
J = F.jacobian([x, y])
print("fixed points:", fps)
for p in fps:
    Jp = np.array(J.subs(p), dtype=float)
    ev, kind = cb.classify_fixed_point(Jp)
    print(p, "eigenvalues", np.round(ev, 4), "->", kind)

In [ ]:
f = sp.lambdify((x, y), list(F), "numpy")
rhs_a = lambda t, s: f(*s)
cb.phase_portrait(rhs_a, (-4, 4), (-4, 4), n=21, style="quiver", nullclines=True)
plt.show()

## Exercise: rabbits versus sheep

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/cont2d.html#rabbits-versus-sheep).
Fixed points symbolically (part b), then phase portraits for the four parameter
choices of part (c).

In [ ]:
x1, x2, r1, r2, b1, b2 = sp.symbols("x1 x2 r1 r2 b1 b2")
G = sp.Matrix([r1 * x1 * (1 - x1) - b1 * x1 * x2,
               r2 * x2 * (1 - x2) - b2 * x1 * x2])
fps = sp.solve(list(G), [x1, x2], dict=True)
for p in fps:
    print(p)
Jg = G.jacobian([x1, x2])

In [ ]:
def rabbits_sheep(t, s, b1=0.0, b2=0.0, r1=3.0, r2=2.0):
    x1, x2 = s
    return [r1 * x1 * (1 - x1) - b1 * x1 * x2, r2 * x2 * (1 - x2) - b2 * x1 * x2]

ics = [[0.1, 1.3], [0.1, 0.2], [1.2, 0.2], [1.2, 1.3]]
fig, axs = plt.subplots(2, 2, figsize=(8, 8))
for ax, (b1v, b2v) in zip(axs.flat, [(0, 0), (1, 2), (3, 1), (4, 3)]):
    cb.phase_portrait(rabbits_sheep, (0, 1.5), (0, 1.5), ax=ax, n=15, style="quiver",
                      trajectories=ics, tmax=30, colors=["r"], b1=b1v, b2=b2v)
    ax.set_title(f"$b_1={b1v}$, $b_2={b2v}$")
fig.tight_layout()
plt.show()

In [ ]:
# linear stability of every fixed point for b1 = 1, b2 = 2 (part d)
vals = {r1: 3, r2: 2, b1: 1, b2: 2}
for p in fps:
    Jp = np.array(Jg.subs(p).subs(vals), dtype=float)
    print({k: float(v.subs(vals)) for k, v in p.items()}, cb.classify_fixed_point(Jp))

# part (e): b1 = 1, scan b2 and watch the eigenvalues of the coexistence point
for b2v in [0.5, 1.0, 1.5, 2.0, 3.0]:
    q = max(fps, key=lambda s: len(str(s)))      # the coexistence point
    Jq = np.array(Jg.subs(q).subs({r1: 3, r2: 2, b1: 1, b2: b2v}), dtype=float)
    print("b2 =", b2v, np.round(np.linalg.eigvals(Jq), 3))

## Exercise: predator-prey with Allee effect

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/cont2d.html#predator-prey-with-alee-effect).
The simplified system with $b=1/2$ is set up below: the non-zero fixed point,
its stability matrix and characteristic polynomial, and a first phase portrait
for $a = 0.75$.

In [ ]:
x, y, a, lam = sp.symbols("x y a lambda", positive=True)
b = sp.Rational(1, 2)
H = sp.Matrix([x**2 / (x + b) - x * y / a, x * y - y**2])
fp = sp.solve(list(H), [x, y], dict=True)
print(fp)                                    # non-zero fixed point: x = y = a - b
J = H.jacobian([x, y]).subs({x: a - b, y: a - b})
print(sp.simplify(J))
print(sp.factor(J.charpoly(lam).as_expr()))
print([sp.simplify(e) for e in J.eigenvals()])

In [ ]:
def allee(t, s, a=0.75, b=0.5):
    x, y = s
    return [x**2 / (x + b) - x * y / a, x * y - y**2]

cb.phase_portrait(allee, (0, 1.5), (0, 1.5), n=21, style="quiver",
                  trajectories=[[0.4, 0.25]], tmax=200, a=0.75)
plt.show()

In [ ]:
# part (g): increase the predators by delta and see whether the run survives
def final_state(delta, tmax=1000):
    sol = solve_ivp(lambda t, s: allee(t, s, a=0.75), [0, tmax], [0.4, 0.25 + delta],
                    rtol=1e-9, atol=1e-12)
    return sol.y[:, -1]

for delta in [0.0, 0.2, 0.4, 0.5, 0.75]:     # refine around the switch-over
    print(delta, np.round(final_state(delta), 4))

## Exercise: nuclear reactor cooling

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/cont2d.html#nuclear-reactor-cooling).
The rescaled system $\dot x = p^2 y - x$, $\dot y = 1 - xy$: fixed point,
trace and determinant of the Jacobian, and the two phase portraits of part (c).

In [ ]:
x, y, p = sp.symbols("x y p", positive=True)
R = sp.Matrix([p**2 * y - x, 1 - x * y])
fp = sp.solve(list(R), [x, y], dict=True)
print(fp)
J = R.jacobian([x, y])
print("trace:", J.trace(), "  det:", sp.simplify(J.det()))
Jfp = sp.simplify(J.subs(fp[0]))
print("eigenvalues at the fixed point:", [sp.simplify(e) for e in Jfp.eigenvals()])
print("discriminant zero at p =", sp.solve(sp.discriminant(Jfp.charpoly().as_expr(), sp.Symbol("lambda")), p))

In [ ]:
def reactor(t, s, p=3.0):
    x, y = s
    return [p**2 * y - x, 1 - x * y]

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, pv in zip(axs, [3.0, 10.0]):
    cb.phase_portrait(reactor, (0, 2 * pv), (0, 3 / pv), ax=ax, n=21, style="quiver",
                      trajectories=[[0.0, 0.0], [2 * pv, 2 / pv]], tmax=50,
                      colors=["b", "r"], p=pv)
    ax.set_title(f"$p={pv:g}$")
fig.tight_layout()
plt.show()

In [ ]:
# parts (d) and (e): curl and divergence of the vector field
fx, fy = R
print("df/dy - dg/dx =", sp.diff(fx, y) - sp.diff(fy, x), "   (zero would mean a gradient system)")
print("div =", sp.diff(fx, x) + sp.diff(fy, y))